# Local LLM Server Check

Run this before the AgentForge/SWE-bench notebook. It checks both `/v1/models` and a tiny `/v1/chat/completions` request so you can separate server issues from mini-swe-agent issues.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

def clean_url(value):
    value = (value or 'http://127.0.0.1:8000/v1').strip()
    if value.startswith('[') and '](' in value and value.endswith(')'):
        value = value.split('](', 1)[1][:-1].strip()
    if value.startswith('<') and value.endswith('>'):
        value = value[1:-1].strip()
    return value.rstrip('/')

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), ROOT

env = os.environ.copy()
env['LLM_BASE_URL'] = clean_url(env.get('LLM_BASE_URL', 'http://127.0.0.1:8000/v1'))
env.setdefault('LLM_API_KEY', 'local')
env.setdefault('UV_CACHE_DIR', str(ROOT / '.uv-cache'))

print(f'Repo: {ROOT}')
print(f"LLM_BASE_URL: {env['LLM_BASE_URL']}")

By default this checks `mlx-community/Qwen2.5-Coder-7B-Instruct-4bit`, the local smoke-test model. Override `MODEL` or `CHECK_MODEL` if you deliberately start a different server model.

In [ ]:
MODEL = os.environ.get('CHECK_MODEL') or os.environ.get('MODEL') or 'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit'
TIMEOUT = os.environ.get('LLM_CHECK_TIMEOUT', '600')

cmd = [
    sys.executable,
    str(ROOT / 'scripts' / 'check_local_llm.py'),
    '--base-url',
    env['LLM_BASE_URL'],
    '--api-key',
    env.get('LLM_API_KEY', 'local'),
    '--timeout',
    TIMEOUT,
    '--max-tokens',
    '8',
    '--print-payload',
]
cmd.extend(['--model', MODEL])

print(shlex.join(cmd))
result = subprocess.run(cmd, cwd=ROOT, env=env, check=False)
if result.returncode != 0:
    raise SystemExit(
        'LLM check failed. Make sure scripts/serve_local_llm.sh is running and wait '
        'until it prints: Local LLM ready at http://127.0.0.1:8000/v1. '
        'If it is still checking generation, wait rather than starting this notebook check again.'
    )